# Prepare pilot simulation data

Export period-mean surface chlorophyll, depth-integrated primary production and control fCO2 from
distributed Julia/JLD2 output. Continuation records are joined to their parent runs using `experiment.txt`.
The output is `data/pilot_2000-2001.nc`, read by `oae_nn_emulator.ipynb`.

The default selection contains Reference and three sensitivity cases (B–D) for 2000–2001.
The sensitivity cases vary the plankton chlorophyll ratio together with the light-attenuation chlorophyll ratio.
The selected cases must cover the same complete period.

fCO2 is calculated from saved control DIC1, Alk1, temperature and salinity using OceanBioME v0.18.0,
at each grid cell and timestamp before time averaging. This step can take substantially longer than NN training.

## 1. Configuration

Run with the model's Julia environment, including JLD2, Oceananigans, OceanBioME v0.18.0 and NCDatasets.
Set the model directory, project directory, source file, selected runs and dates below.

In [11]:
using Dates, Pkg
# Directory containing the simulation output and model source.
model_root = "/nfs/roberts/pi/pi_ey239/qi/global-ocean-model"
# Set the project directory explicitly to override automatic discovery.
project_dir = nothing
chemistry_source = joinpath(model_root, "src", "Biogeochemistry_bio.jl")
# OceanBioME v0.18.0 defaults for parameters absent from Reference metadata.
reference_defaults = (chlorophyll_ratio=1.31, light_half_saturation=33.0)
period_start = Date(2000, 1, 1)
period_end = Date(2001, 12, 31)  # Inclusive; every selected case needs this complete period.
day_zero = Date(2000, 1, 1)
cases = [
    (id="reference", folder="output_bio_pp", leaf="bio_reference_continue_28390280"),
    (id="chl30_kfe50", folder="output_bio_pp_abcd", leaf="bio_pp_abcd_chl30_kfe50_continue_28506337"),
    (id="chl30_growth_plus25", folder="output_bio_pp_abcd", leaf="bio_pp_abcd_chl30_growth_plus25_continue_28506338"),
    (id="chl30_kfe50_growth_plus25", folder="output_bio_pp_abcd", leaf="bio_pp_abcd_chl30_kfe50_growth_plus25_continue_28506339"),
]

if isnothing(project_dir)
    candidates = String[]
    cursor = abspath(pwd())
    while true
        push!(candidates, cursor)
        dirname(cursor) == cursor && break
        cursor = dirname(cursor)
    end
    qz_roots = [p for p in candidates if basename(p) == "qz334" && isdir(p)]
    isdir(joinpath(pwd(),"qz334")) && push!(qz_roots,joinpath(pwd(),"qz334"))
    isempty(qz_roots) && error("Set project_dir to the project directory, or start from qz334/ML4OAE.")
    project_dir = joinpath(first(qz_roots),"ML4OAE")
end
isdir(project_dir) || error("ML4OAE folder does not exist: $project_dir")
data_dir = joinpath(project_dir,"data")
mkpath(data_dir)
Pkg.activate(model_root)
using JLD2, Oceananigans, OceanBioME, NCDatasets
println("Read model outputs from: ", model_root)
println("Write pilot inputs to: ", data_dir)
println("fCO2 calculation uses ", Threads.nthreads(), " Julia thread(s) for the saved surface fields.")


  Activating project at `/nfs/roberts/pi/pi_ey239/qi/global-ocean-model`


Read model outputs from: /nfs/roberts/pi/pi_ey239/qi/global-ocean-model
Write pilot inputs to: /nfs/roberts/pi/pi_ey239/qz334/ML4OAE/data
fCO2 calculation uses 1 Julia thread(s) for the saved surface fields.


## 2. Output readers

The helper functions merge restart records, pair MPI ranks, remove halos and read native-grid geometry.
PP uses complete daily means; CHL and fCO2 use time-weighted surface snapshots.

In [12]:
module BiologyCatalog

export BiologyRun, discover_biology_runs, resolve_biology_lineage,
       select_biology_lineage, stitch_biology_records, validate_biology_window,
       surface_file_parts, read_biology_metadata, biology_restart_boundaries

"""A run is one fresh or continued process, not an entire experiment."""
struct BiologyRun
    name::String
    directory::String
    case::String
    metadata::Dict{String,String}
    parent::Union{Nothing,String}
    restored_day::Union{Nothing,Float64}
    parameter_signature::String
    continuation::Bool
end

compact_parameter_text(s) = filter(c -> !isspace(c), String(s))

function read_biology_metadata(path)
    isfile(path) || error("Missing experiment metadata: $path")
    data = Dict{String,String}()
    for line in eachline(path)
        pieces = split(line, '='; limit=2)
        length(pieces) == 2 || continue
        key, value = strip.(pieces)
        if haskey(data, key) && data[key] != value &&
                key in ("case", "run_name", "plankton_parameters", "resume_source", "restored_day",
                        "resume_iteration", "restored_iteration", "pp_diagnostics_schema")
            error("Conflicting repeated metadata key '$key' in $path")
        end
        data[key] = value
    end
    return data
end

function directory_case(name, case_ids)
    for case in case_ids
        for prefix in ("bio_$(case)_", "bio_pp_abcd_$(case)_")
            startswith(name, prefix) || continue
            suffix = name[length(prefix)+1:end]
            occursin(r"^(continue_)?[0-9]+$", suffix) && return String(case)
        end
    end
    return nothing
end

"""
    discover_biology_runs(output_root, case_ids; expected_parameters=Dict())

Read metadata without evaluating Julia text. Return `(runs, invalid)`, where
`runs` maps exact run directory names to `BiologyRun` and `invalid` contains
`(name, directory, case, reason)` entries. Failed continuations lacking a
successful restored time are quarantined, not silently used as fresh runs.
Expected parameter signatures are optional and may be strings or NamedTuples.
"""
function discover_biology_runs(output_root, case_ids; expected_parameters=Dict())
    isdir(output_root) || error("Output directory does not exist: $output_root")
    ids = String.(collect(case_ids))
    length(unique(ids)) == length(ids) || error("Duplicate case IDs")
    runs = Dict{String,BiologyRun}()
    invalid = NamedTuple[]
    for directory in sort(readdir(output_root; join=true))
        isdir(directory) || continue
        name = basename(directory)
        case = directory_case(name, ids)
        isnothing(case) && continue
        try
            metadata = read_biology_metadata(joinpath(directory, "experiment.txt"))
            get(metadata, "case", "") == case || error("Directory/metadata case mismatch")
            get(metadata, "run_name", "") == name || error("Directory/metadata run_name mismatch")
            parameters = compact_parameter_text(get(metadata, "plankton_parameters", ""))
            isempty(parameters) && error("Missing plankton_parameters")
            if haskey(expected_parameters, case) || haskey(expected_parameters, Symbol(case))
                expected = get(expected_parameters, case, get(expected_parameters, Symbol(case), nothing))
                expected_text = expected isa AbstractString ? expected : repr(expected)
                parameters == compact_parameter_text(expected_text) || error("Parameters do not match expected case definition")
            end
            continuation = startswith(name, "bio_$(case)_continue_") ||
                           startswith(name, "bio_pp_abcd_$(case)_continue_")
            parent = nothing
            restored_day = nothing
            if continuation
                get(metadata, "mode", "") == "continuation" || error("Continuation directory lacks mode = continuation")
                get(metadata, "restore_status", "") == "successful" || error("Checkpoint restore is not recorded as successful")
                source = get(metadata, "resume_source", "")
                isempty(source) && error("Continuation lacks resume_source")
                parent = basename(normpath(source))
                directory_case(parent, [case]) == case || error("resume_source names a different or invalid case")
                restored_day = tryparse(Float64, get(metadata, "restored_day", ""))
                isnothing(restored_day) && error("Continuation lacks a numeric restored_day")
                isfinite(restored_day) && restored_day >= 0 || error("Invalid restored_day")
            else
                haskey(metadata, "resume_source") && error("Fresh run directory unexpectedly declares resume_source")
                get(metadata, "mode", "fresh") != "continuation" || error("Fresh run directory declares continuation mode")
            end
            runs[name] = BiologyRun(name, abspath(directory), case, metadata,
                                     parent, restored_day, parameters, continuation)
        catch err
            push!(invalid, (; name, directory=abspath(directory), case,
                             reason=sprint(showerror, err)))
        end
    end
    return (; runs, invalid)
end

"""Follow recorded parent links; never join independent fresh runs."""
function resolve_biology_lineage(runs, leaf_name)
    name = String(leaf_name)
    haskey(runs, name) || error("Unknown or invalid selected run: $name")
    chain = BiologyRun[]
    seen = Set{String}()
    leaf = runs[name]
    while true
        name in seen && error("Cycle in restart lineage at $name")
        push!(seen, name)
        haskey(runs, name) || error("Missing or invalid lineage ancestor: $name")
        run = runs[name]
        run.case == leaf.case || error("Case changes within restart lineage")
        run.parameter_signature == leaf.parameter_signature || error("Biology parameters change within restart lineage at $name")
        push!(chain, run)
        isnothing(run.parent) && break
        name = run.parent
    end
    reverse!(chain)
    for i in 2:length(chain)
        parent, child = chain[i-1], chain[i]
        isnothing(child.restored_day) && error("Missing restored_day for $(child.name)")
        if !isnothing(parent.restored_day) && child.restored_day < parent.restored_day
            error("Restart time goes backwards within lineage: $(child.name)")
        end
    end
    return chain
end

"""
Choose a unique terminal run or an explicit leaf. Multiple fresh reruns or
continuation branches require an override; later job IDs never imply validity.
An explicit ancestor is allowed for audit/recovery and is not silently extended.
"""
function select_biology_lineage(runs, case_id; leaf_override=nothing)
    case = String(case_id)
    if !isnothing(leaf_override) && !isempty(String(leaf_override))
        name = basename(normpath(String(leaf_override)))
        haskey(runs, name) || error("Selected run is missing or invalid: $name")
        runs[name].case == case || error("Selected run does not belong to case $case")
        return resolve_biology_lineage(runs, name)
    end
    candidates = [r for r in values(runs) if r.case == case]
    isempty(candidates) && error("No valid runs found for case $case")
    parents = Set(r.parent for r in candidates if !isnothing(r.parent))
    terminals = sort([r.name for r in candidates if !(r.name in parents)])
    length(terminals) == 1 || error("Case $case has $(length(terminals)) terminal runs; set a leaf override: $(join(terminals, ", "))")
    return resolve_biology_lineage(runs, only(terminals))
end

"""
Inventory output files by MPI rank. Accept both `_rank0_part1.jld2` and
`_part1_rank0.jld2`, plus unsplit `_rank0.jld2`. Rank snapshots must be paired
by time by the reader, not by part number: a process may split at another time.
"""
function surface_file_parts(run::BiologyRun; prefix="surface_fields", nranks=2, allow_missing=false)
    nranks >= 1 || error("nranks must be positive")
    base = "$(prefix)_$(run.name)"
    inventory = Dict{Int,Vector{Tuple{Int,String}}}()
    for path in readdir(run.directory; join=true)
        isfile(path) || continue
        file = basename(path)
        startswith(file, base * "_") && endswith(file, ".jld2") || continue
        suffix = file[length(base)+1:end]
        first_pattern = match(r"^_rank([0-9]+)(?:_part([0-9]+))?\.jld2$", suffix)
        second_pattern = match(r"^_part([0-9]+)_rank([0-9]+)\.jld2$", suffix)
        rank, part = if !isnothing(first_pattern)
            (parse(Int, first_pattern.captures[1]), isnothing(first_pattern.captures[2]) ? 0 : parse(Int, first_pattern.captures[2]))
        elseif !isnothing(second_pattern)
            (parse(Int, second_pattern.captures[2]), parse(Int, second_pattern.captures[1]))
        else
            error("Unrecognized surface output filename: $file")
        end
        0 <= rank < nranks || error("Unexpected MPI rank $rank in $file")
        filesize(path) > 0 || error("Empty output file: $path")
        push!(get!(inventory, rank, Tuple{Int,String}[]), (part, path))
    end
    (allow_missing || Set(keys(inventory)) == Set(0:nranks-1)) ||
        error("Missing surface output rank(s) in $(run.name)")
    for (rank, parts) in inventory
        length(unique(first.(parts))) == length(parts) || error("Duplicate file part numbers for rank $rank in $(run.name)")
        sort!(parts; by=first)
    end
    return Dict(rank => last.(get(inventory, rank, Tuple{Int,String}[])) for rank in 0:nranks-1)
end

# A checkpoint-boundary sample can be inherited only when its paired iteration
# keys identify the exact state restored by the continuation. File times alone
# are insufficient evidence, and metadata conflicts must not be guessed away.
function checkpoint_iteration(run::BiologyRun)
    values = Dict{String,Int}()
    for key in ("restored_iteration", "resume_iteration")
        haskey(run.metadata, key) || continue
        value = tryparse(Int, strip(run.metadata[key]))
        isnothing(value) && error("Invalid $key in $(run.name)")
        value >= 0 || error("Invalid $key in $(run.name)")
        values[key] = value
    end
    if length(values) == 2 && values["restored_iteration"] != values["resume_iteration"]
        error("Conflicting restored_iteration and resume_iteration in $(run.name)")
    end
    return get(values, "restored_iteration", get(values, "resume_iteration", nothing))
end

function checkpoint_record_matches(row, iteration)
    isnothing(iteration) && return false
    hasproperty(row, :keys) || return false
    keys = row.keys
    (keys isa Tuple || keys isa AbstractVector) && !isempty(keys) || return false
    return all(key -> tryparse(Int, string(key)) == iteration, keys)
end

"""
    stitch_biology_records(lineage, records_by_run; day_of=r->r.time/86400)

Records supplied by the reader must already pair all MPI ranks at each time.
Parent snapshots strictly after a child's restored_day are discarded, including
parent-only snapshots after a rollback. At the checkpoint itself a paired parent
snapshot is retained only when its iteration keys match the child's recorded
restored/resume iteration. If the child saved that time, its snapshot takes
precedence. Missing samples after the checkpoint are never filled from a parent.
The output records retain their fields and add/replace `run_name` provenance.
"""
function stitch_biology_records(lineage, records_by_run; day_of=r -> r.time/86400,
                                tolerance_seconds=1.0)
    isempty(lineage) && error("Empty lineage")
    tolerance_seconds >= 0 || error("Negative time tolerance")
    tolerance = tolerance_seconds/86400
    stitched = NamedTuple[]
    for (i, run) in enumerate(lineage)
        haskey(records_by_run, run.name) || error("No paired record catalog for $(run.name)")
        records = sort(collect(records_by_run[run.name]); by=day_of)
        previous_day = -Inf
        for row in records
            day = Float64(day_of(row))
            isfinite(day) && day >= 0 || error("Invalid record time in $(run.name)")
            day > previous_day + tolerance || error("Duplicate or ambiguous times within $(run.name)")
            previous_day = day
        end
        if i == 1
            isnothing(run.restored_day) || error("Lineage must begin with a fresh run")
        else
            boundary = run.restored_day
            isnothing(boundary) && error("Missing child restored_day")
            iteration = checkpoint_iteration(run)
            child_boundary = [row for row in records if abs(day_of(row)-boundary) <= tolerance]
            length(child_boundary) <= 1 || error("Ambiguous checkpoint-boundary times in $(run.name)")
            # Keep the trajectory before the checkpoint. At the checkpoint,
            # child output wins; otherwise a verified inherited state is valid.
            # This also preserves provenance through restarts at the same time.
            filter!(stitched) do row
                day = day_of(row)
                day < boundary-tolerance ||
                    (isempty(child_boundary) && abs(day-boundary) <= tolerance &&
                     checkpoint_record_matches(row, iteration))
            end
            records = [row for row in records if day_of(row) >= boundary-tolerance]
        end
        append!(stitched, [merge(row, (; run_name=run.name)) for row in records])
    end
    sort!(stitched; by=day_of)
    days = Float64[day_of(row) for row in stitched]
    all(diff(days) .> tolerance) || error("Ambiguous duplicate times in stitched lineage")
    return stitched
end

"""
    biology_restart_boundaries(lineage, records_by_run, stitched)

Report raw paired parent/child samples around each checkpoint boundary and the
sample retained in the final stitched trajectory. A boundary retained from an
ancestor is explicitly identified by `selected_run`. Missing raw samples remain
missing; this diagnostic neither reads field arrays nor synthesizes output.
"""
function biology_restart_boundaries(lineage, records_by_run, stitched;
                                    day_of=r -> r.time/86400, tolerance_seconds=1.0)
    tolerance_seconds >= 0 || error("Negative time tolerance")
    tolerance = tolerance_seconds/86400
    reports = NamedTuple[]
    for i in 2:length(lineage)
        parent, child = lineage[i-1], lineage[i]
        boundary = child.restored_day
        isnothing(boundary) && error("Missing child restored_day")
        iteration = checkpoint_iteration(child)
        parent_rows = records_by_run[parent.name]
        child_rows = records_by_run[child.name]
        at_boundary(rows) = [r for r in rows if abs(day_of(r)-boundary) <= tolerance]
        parent_at, child_at, selected = at_boundary(parent_rows), at_boundary(child_rows), at_boundary(stitched)
        all(length(v) <= 1 for v in (parent_at, child_at, selected)) ||
            error("Ambiguous checkpoint-boundary diagnostic at $(child.name)")
        boundary_keys(rows) = isempty(rows) ? "" :
            hasproperty(only(rows), :keys) ? join(string.(only(rows).keys), "|") : "unavailable"
        function previous_day(rows)
            days = [Float64(day_of(r)) for r in rows if day_of(r) < boundary-tolerance]
            return isempty(days) ? missing : maximum(days)
        end
        function next_day(rows)
            days = [Float64(day_of(r)) for r in rows if day_of(r) > boundary+tolerance]
            return isempty(days) ? missing : minimum(days)
        end
        verified = !isempty(parent_at) && checkpoint_record_matches(only(parent_at), iteration)
        selected_run = isempty(selected) ? "" : only(selected).run_name
        status = if selected_run == child.name
            "child checkpoint snapshot selected"
        elseif !isempty(selected_run) && verified && selected_run == parent.name
            "verified parent checkpoint snapshot retained"
        elseif !isempty(selected_run) && checkpoint_record_matches(only(selected), iteration) &&
                selected_run in getfield.(lineage[1:i-1], :name)
            "verified ancestor checkpoint snapshot retained"
        elseif !isempty(selected_run)
            "checkpoint snapshot selected from a later lineage segment"
        elseif isempty(parent_at) && isempty(child_at)
            "checkpoint snapshot absent from both raw segments"
        elseif !isempty(parent_at) && isempty(child_at) && isnothing(iteration)
            "parent checkpoint snapshot not retained: iteration metadata unavailable"
        elseif !isempty(parent_at) && isempty(child_at) && !verified
            "parent checkpoint snapshot not retained: iteration keys do not match"
        else
            "checkpoint snapshot superseded by a later restart"
        end
        push!(reports, (; parent_run=parent.name, child_run=child.name,
            restored_day=boundary, restored_iteration=isnothing(iteration) ? missing : iteration,
            parent_boundary_present=!isempty(parent_at), child_boundary_present=!isempty(child_at),
            parent_boundary_iteration_verified=verified,
            parent_boundary_keys=boundary_keys(parent_at), child_boundary_keys=boundary_keys(child_at),
            parent_previous_day=previous_day(parent_rows), parent_next_day=next_day(parent_rows),
            child_previous_day=previous_day(child_rows), child_next_day=next_day(child_rows),
            selected_run, status))
    end
    return reports
end

"""
Check observed coverage without inferring success from directory names or Slurm
exit state. Boundary bracketing is allowed for trapezoidal interpolation; all
gaps intersecting the requested window must be at most max_gap_days. Call with
0–731 for the complete trajectory, or 366–731 for the post-addition analysis.
"""
function validate_biology_window(records; start_day=366.0, stop_day=731.0,
                                  max_gap_days=1.05, tolerance_seconds=1.0,
                                  day_of=r -> r.time/86400)
    isfinite(start_day) && isfinite(stop_day) && stop_day > start_day || error("Invalid analysis window")
    max_gap_days > 0 && tolerance_seconds >= 0 || error("Invalid gap/tolerance setting")
    tolerance = tolerance_seconds/86400
    problems = String[]
    days = Float64[day_of(row) for row in records]
    if isempty(days)
        return (; complete=false, first_day=NaN, last_day=NaN,
                 maximum_gap_days=Inf, n_records=0, problems=["No paired output records"])
    end
    if !all(isfinite, days) || !all(diff(days) .> tolerance)
        return (; complete=false, first_day=first(days), last_day=last(days),
                 maximum_gap_days=Inf, n_records=length(days),
                 problems=["Output times must be finite, strictly increasing and unambiguous"])
    end
    first(days) <= start_day+tolerance || push!(problems, "Output begins at day $(first(days)), after required day $start_day")
    last(days) >= stop_day-tolerance || push!(problems, "Output ends at day $(last(days)), before required day $stop_day")
    gaps = Float64[]
    for i in 1:length(days)-1
        # Include brackets that straddle a boundary, not unrelated spin-up gaps.
        days[i] < stop_day-tolerance && days[i+1] > start_day+tolerance || continue
        gap = days[i+1]-days[i]
        push!(gaps, gap)
        gap <= max_gap_days+tolerance || push!(problems, "Gap of $(round(gap; digits=5)) days between model days $(days[i]) and $(days[i+1])")
    end
    isempty(gaps) && push!(problems, "No intervals cover the requested analysis window")
    return (; complete=isempty(problems), first_day=first(days), last_day=last(days),
             maximum_gap_days=isempty(gaps) ? Inf : maximum(gaps),
             n_records=length(days), problems)
end

end # module


module BiologyAnalysis
using JLD2, Oceananigans
using Oceananigans: Center, Face
using Oceananigans.Architectures: CPU, architecture, on_architecture
using Oceananigans.Grids: inactive_cell, topology, rnode
using Oceananigans.ImmersedBoundaries: ImmersedBoundaryGrid, GridFittedBottom
const grid_cache = IdDict{Any, Dict{String, Any}}()

function preserved_cpu_grid(saved_grid)
    if saved_grid isa ImmersedBoundaryGrid
        underlying = saved_grid.underlying_grid
        grid = architecture(underlying) isa CPU ? underlying : on_architecture(CPU(), underlying)
        boundary = on_architecture(CPU(), saved_grid.immersed_boundary)
        TX, TY, TZ = topology(saved_grid)
        # The ordinary architecture conversion reconstructs the immersed grid.
        # Preserve its saved bottom values, including rank-edge cells, and omit
        # unused device cell maps.
        return ImmersedBoundaryGrid{TX, TY, TZ}(grid, boundary, nothing, nothing)
    end
    return architecture(saved_grid) isa CPU ? saved_grid : on_architecture(CPU(), saved_grid)
end


function read_geometry(file, variable)
    prefix = "timeseries/$variable/serialized"
    p = haskey(file, "$prefix/grid_index") ? "serialized/grid_$(file["$prefix/grid_index"])" : "serialized/grid"
    cache = get!(() -> Dict{String, Any}(), grid_cache, file)
    geometry = get!(cache, p) do
        saved_grid = file[p]
        grid = preserved_cpu_grid(saved_grid)
        g = hasproperty(grid, :underlying_grid) ? grid.underlying_grid : grid
        nx, ny, nz = Int(g.Nx), Int(g.Ny), Int(g.Nz)
        lon = vec(Float64.(Array(g.λᶜᶜᵃ[1:nx, 1:ny])))
        lat = vec(Float64.(Array(g.φᶜᶜᵃ[1:nx, 1:ny])))
        wet = vec([!inactive_cell(i, j, nz, grid) for i in 1:nx, j in 1:ny])
        bottom = fill(NaN, nx * ny)
        if grid isa ImmersedBoundaryGrid && grid.immersed_boundary isa GridFittedBottom
            boundary = grid.immersed_boundary
            bottom = vec([Float64(boundary.bottom_height[i, j, 1]) for i in 1:nx, j in 1:ny])
            all(isfinite, bottom) || error("Saved bottom height contains nonfinite interior values")
            physical_wet = vec([
                rnode(i, j, nz, g, Center(), Center(), Center()) > boundary.bottom_height[i, j, 1] &&
                !inactive_cell(i, j, nz, g) for i in 1:nx, j in 1:ny])
            wet == physical_wet || error("Surface wet mask disagrees with saved bathymetry")
        end
        all(isfinite, lon) && all(y -> isfinite(y) && -90 <= y <= 90, lat) || error("Invalid grid coordinates")
        (; nx, ny, nz, grid, lon=mod.(lon .+ 180, 360) .- 180, lat, wet, bottom)
    end
    return (; geometry...,
            location=file["$prefix/location"], indices=file["$prefix/indices"])
end


function verify_geometry(a, b)
    (a.nx, a.ny, a.nz) == (b.nx, b.ny, b.nz) && a.wet == b.wet || error("Different grids between runs")
    maximum(abs.(mod.(a.lon .- b.lon .+ 180, 360) .- 180)) <= 1e-6 &&
        maximum(abs.(a.lat .- b.lat)) <= 1e-6 || error("Different coordinates between runs")
    isequal(a.bottom, b.bottom) || error("Different saved bathymetry between inputs")
end


function horizontal_slice(raw, n, dim, loc, ind)
    face = loc === Face || loc isa Face
    center = loc === Center || loc isa Center
    face || center || error("Unsupported horizontal field location: $loc")
    extra = face ? mod(size(raw, dim)-n, 2) : 0
    padding = size(raw, dim)-n-extra
    padding >= 0 && iseven(padding) || error("Incompatible saved field shape")
    halo = padding ÷ 2
    if !(ind isa Colon)
        ind isa AbstractUnitRange && first(ind) == 1-halo && length(ind) == size(raw, dim) ||
            error("Unsupported partial horizontal slice: $ind")
    end
    stop = halo+n+Int(face)
    stop <= size(raw, dim) || error("Face interpolation needs the saved halo or boundary face")
    return (halo+1:stop), face
end


function surface_values(file, variable, key, g)
    raw = Array(file["timeseries/$variable/$key"])
    ndims(raw) == 3 && size(raw, 3) == 1 && (raw = dropdims(raw; dims=3))
    ndims(raw) == 2 || error("$variable must be a single saved layer, got $(size(raw))")
    z = g.indices[3]
    if !(g.location[3] === Nothing || g.location[3] isa Nothing)
        # Eta is saved at the top face; the ocean writer slices at k = Nz.
        top_face = variable == "η" && (g.location[3] === Face || g.location[3] isa Face)
        expected_z = top_face ? g.nz+1 : g.nz
        (z == expected_z || (z isa AbstractUnitRange && first(z) == last(z) == expected_z)) ||
            error("$variable must be saved at k = $expected_z; got $z")
    end
    ix, xface = horizontal_slice(raw, g.nx, 1, g.location[1], g.indices[1])
    iy, yface = horizontal_slice(raw, g.ny, 2, g.location[2], g.indices[2])
    values = Float64.(raw[ix, iy])
    # Move horizontal face velocities to cell centers; keep grid-aligned components.
    xface && (values = (values[1:end-1, :] .+ values[2:end, :]) ./ 2)
    yface && (values = (values[:, 1:end-1] .+ values[:, 2:end]) ./ 2)
    return vec(values)
end


function with_input_files(fn, records)
    files = Dict{String,Any}()
    try
        for path in unique([p for r in records for p in r.paths])
            files[path] = jldopen(path,"r";iotype=IOStream)
        end
        return fn(files)
    finally
        for f in values(files)
            try close(f) catch end
            delete!(grid_cache,f)
        end
    end
end


function file_geometry(files,path,v,cache)
    get!(cache,(path,v)) do
        read_geometry(files[path],v)
    end
end


end
module BiologyChlorophyll
const seconds_per_day = 86400.0

function window_records(records, start, stop; max_gap=1.05*seconds_per_day)
    isempty(records) && error("No chlorophyll snapshots")
    times = Float64[r.time for r in records]
    all(isfinite, times) && all(diff(times) .> 0) || error("Snapshot timestamps must be finite and strictly increasing")
    start < stop || error("Invalid chlorophyll month interval")
    lo, hi = searchsortedlast(times, start), searchsortedfirst(times, stop)
    lo >= 1 && hi <= length(times) || error("Chlorophyll month lacks actual bracketing snapshots: days $(start/seconds_per_day)–$(stop/seconds_per_day)")
    selected = records[lo:hi]
    all(diff(times[lo:hi]) .<= max_gap) || error("Missing daily chlorophyll interval (> $(max_gap/seconds_per_day) days); comparison blocked for all cases")
    return selected
end


function stream_month_mean(read_snapshot, records, start, stop; max_gap=1.05*seconds_per_day)
    selected = window_records(records, start, stop; max_gap)
    previous = Float64.(read_snapshot(first(selected)))
    integral = zeros(length(previous))
    covered = 0.0
    for k in 2:length(selected)
        current = Float64.(read_snapshot(selected[k]))
        length(current) == length(previous) || error("Chlorophyll grid size changed inside a month")
        t0, t1 = selected[k-1].time, selected[k].time
        left, right = max(start, t0), min(stop, t1)
        if right > left
            a, b = (left-t0)/(t1-t0), (right-t0)/(t1-t0)
            # Weighted endpoints preserve NaNs, so incomplete pixels cannot
            # receive an apparently complete monthly mean.
            integral .+= ((1-a).*previous .+ a.*current .+
                         (1-b).*previous .+ b.*current) .* ((right-left)/2)
            covered += right-left
        end
        previous = current
    end
    isapprox(covered, stop-start; atol=1e-6, rtol=1e-12) || error("Chlorophyll month is not fully covered")
    return integral ./ (stop-start)
end


end
module DailyPPReader
using JLD2
using Oceananigans: Center
import ..BiologyCatalog
import ..BiologyAnalysis
const day_seconds = 86400.0
const time_tolerance = 1.0
const column_variables = ("PP_C_1pct", "PP_C_01pct", "PP_C_full_column")
const required_fields = (column_variables..., "PP_invalid_fraction")
stamp(path) = (; path=abspath(path), size=filesize(path), mtime=stat(path).mtime)

function validate_input_options(input_mode, cutoff_day=nothing)
    input_mode in (:strict, :fixed_records) || error("input_mode must be :strict or :fixed_records")
    if !isnothing(cutoff_day)
        cutoff_day isa Real && isfinite(cutoff_day) && cutoff_day >= 0 ||
            error("cutoff_day must be a finite nonnegative model day")
    end
    return nothing
end


function verify_metadata(file, path; variable="PP_C_surface")
    metadata(key) = haskey(file, "pp_metadata/$key") ? String(file["pp_metadata/$key"]) :
        error("Missing pp_metadata/$key in $path; instantaneous PP cannot be averaged into daily PP after the run")
    schema = metadata("version")
    schema in ("pp_depth_v1", "pp_full_column_v1") || error("Unrecognized daily PP schema in $path")
    if schema == "pp_full_column_v1"
        variable == "PP_C_full_column" || error(
            "Schema pp_full_column_v1 does not save $variable; select PP_C_full_column in $path")
    end
    if haskey(file, "pp_metadata/output_mode")
        expected_mode = schema == "pp_depth_v1" ? "depth" : "full_column"
        metadata("output_mode") == expected_mode || error("Conflicting PP schema/output_mode in $path")
    end
    averaging = replace(metadata("averaging"), " " => "")
    startswith(averaging, "AveragedTimeInterval(1day,window=1day,stride=1)") &&
        occursin("timestampiswindowend", averaging) ||
        error("Daily PP must represent a full one-day averaging window, every step, with window-end times: $path")
    key, expected = if variable == "PP_C_surface"
        ("PP_C_surface_units", "mg C m^-3 day^-1; daily time mean")
    elseif variable in column_variables
        ("PP_C_integral_units", "mg C m^-2 day^-1; daily time mean")
    else
        error("Unsupported daily movie variable $variable")
    end
    metadata(key) == expected || error("Unexpected units for $variable in $path")
    return expected
end


function index_daily_run(run; allow_empty=false, input_mode=:strict, cutoff_day=nothing)
    validate_input_options(input_mode, cutoff_day)
    parts = BiologyCatalog.surface_file_parts(run; prefix="pp_daily", allow_missing=allow_empty)
    paths = sort!(vcat(parts[0], parts[1]))
    before = input_mode == :strict ? stamp.(paths) : nothing
    tables = Dict{Int,Vector{NamedTuple}}()
    seen_schema = nothing
    for rank in 0:1
        bytime = Dict{Int,NamedTuple}()
        for path in parts[rank]
            jldopen(path, "r"; iotype=IOStream) do file
                verify_metadata(file, path; variable="PP_C_full_column")
                schema = String(file["pp_metadata/version"])
                expected_schema = get(run.metadata, "pp_diagnostics_schema", schema)
                schema == expected_schema || error("Run/file PP schema mismatch in $path")
                (isnothing(seen_schema) || seen_schema == schema) || error("PP schema changes within $(run.name)")
                seen_schema = schema
                required = schema == "pp_depth_v1" ? required_fields : ("PP_C_full_column", "PP_invalid_fraction")
                for variable in (schema == "pp_depth_v1" ? column_variables : ("PP_C_full_column",))
                    verify_metadata(file, path; variable)
                end
                allow_empty && !haskey(file, "timeseries/t") && return
                haskey(file, "timeseries/t") || error("No daily PP timestamps in $path")
                saved_keys = String.(collect(keys(file["timeseries/t"])))
                for key in saved_keys
                    isnothing(tryparse(Int, key)) && continue
                    time = Float64(file["timeseries/t/$key"])
                    # A frozen cutoff excludes newer writes before checking their
                    # averaging boundaries or the presence of diagnostic fields.
                    !isnothing(cutoff_day) && time > cutoff_day * day_seconds + time_tolerance && continue
                    isfinite(time) && time >= 0 || error("Invalid PP timestamp in $path")
                    time <= time_tolerance && continue  # initialization, not [−1, 0] mean
                    if !isnothing(run.restored_day) &&
                            abs(time - run.restored_day * day_seconds) <= time_tolerance
                        continue # possibly unfinished restored average / initialization write
                    end
                    day_end = round(Int, time / day_seconds)
                    abs(time - day_end * day_seconds) <= time_tolerance ||
                        error("PP record is not at a full daily boundary in $path at day $(time/day_seconds)")
                    # Incomplete last writes are omitted; month_records then refuses that month.
                    all(haskey(file, "timeseries/$v/$key") for v in required) || continue
                    row = (; time=day_end * day_seconds, key, path)
                    if haskey(bytime, day_end) && bytime[day_end].key != key
                        error("Conflicting iterations for day $day_end in $(run.name), rank $rank")
                    end
                    bytime[day_end] = row # newer split file supersedes an identical saved time
                end
            end
        end
        tables[rank] = sort!(collect(values(bytime)); by=r -> r.time)
    end
    if input_mode == :strict
        isequal(before, stamp.(paths)) || error("Daily PP files changed while indexing $(run.name); retry after output writing stops")
    end
    right = Dict(r.time => r for r in tables[1])
    records = NamedTuple[]
    for a in tables[0]
        haskey(right, a.time) || continue
        b = right[a.time]
        a.key == b.key || error("Daily PP rank iterations disagree on day $(a.time/day_seconds) in $(run.name)")
        push!(records, (; time=a.time, keys=(a.key, b.key), paths=(a.path, b.path)))
    end
    return records
end


function window_records(catalog, start_day::Int, stop_day::Int)
    0 <= start_day < stop_day || error("Daily PP window requires 0 <= start_day < stop_day")
    selected = sort([r for r in catalog.records if
        r.time > start_day * day_seconds + time_tolerance &&
        r.time <= stop_day * day_seconds + time_tolerance]; by=r -> r.time)
    expected = collect(start_day+1:stop_day)
    ends = round.(Int, [r.time / day_seconds for r in selected])
    length(unique(ends)) == length(ends) || error("Duplicate daily PP windows in ($start_day, $stop_day]")
    all(isfinite(r.time) && abs(r.time - day * day_seconds) <= time_tolerance
        for (r, day) in zip(selected, ends)) ||
        error("Off-boundary PP timestamp in ($start_day, $stop_day]")
    ends == expected || error("Incomplete daily PP for ($start_day, $stop_day]; missing window-end model days: $(join(setdiff(expected, ends), ", "))")
    return selected
end


function plane_values(file, variable, key, geometry)
    raw = Array(file["timeseries/$variable/$key"])
    ndims(raw) == 3 && size(raw, 3) == 1 && (raw = dropdims(raw; dims=3))
    ndims(raw) == 2 || error("$variable must be a horizontal diagnostic plane; found $(size(raw))")
    all(loc -> loc === Center || loc isa Center, geometry.location[1:2]) ||
        error("Daily PP requires Center, Center horizontal locations")
    (geometry.location[3] === Nothing || geometry.location[3] isa Nothing) ||
        error("$variable must be a vertically reduced plane (Center, Center, Nothing)")
    ix, xface = BiologyAnalysis.horizontal_slice(raw, geometry.nx, 1, geometry.location[1], geometry.indices[1])
    iy, yface = BiologyAnalysis.horizontal_slice(raw, geometry.ny, 2, geometry.location[2], geometry.indices[2])
    (xface || yface) && error("Daily PP unexpectedly lives on horizontal faces")
    return vec(Float64.(raw[ix, iy]))
end


function average_planes(read_plane, records)
    isempty(records) && error("No daily PP windows")
    total = Float64.(read_plane(first(records)))
    for record in records[2:end]
        values = read_plane(record)
        length(values) == length(total) || error("Daily PP grid changed within the averaging interval")
        total .+= values
    end
    # All selected windows have exactly equal one-day duration. NaNs propagate;
    # an interval mean never conceals an invalid/missing day by renormalization.
    return total ./ length(records)
end


end
module ReferenceChlComparison
using JLD2
import ..BiologyCatalog
const day_seconds = 86400.0
fingerprint(path) = let s = stat(path)
    (; size=s.size, mtime=s.mtime, ctime=s.ctime, inode=s.inode)
end
function assert_unchanged(path, expected)
    isfile(path) && fingerprint(path) == expected || error("CHL input changed: $path")
end

function index_chl_run(run, issues; input_mode=:strict, cutoff_day=nothing)
    input_mode in (:strict, :fixed_records) || error("Unsupported model input mode: $input_mode")
    isnothing(cutoff_day) || (cutoff_day isa Real && isfinite(cutoff_day) && cutoff_day >= 0) ||
        error("CHL cutoff_day must be a finite nonnegative model day or nothing")
    cutoff_time = isnothing(cutoff_day) ? Inf : Float64(cutoff_day) * day_seconds
    parts = try
        BiologyCatalog.surface_file_parts(run)
    catch err
        occursin("Missing surface output rank(s)", sprint(showerror, err)) || rethrow()
        push!(issues, (; run=run.name, rank=-1, path=run.directory, model_day=NaN,
            key="", status="missing surface rank files", detail=sprint(showerror, err)))
        return (; records=NamedTuple[], fingerprints=Dict{String,Any}())
    end
    tables = Dict{Int,Vector{NamedTuple}}()
    fingerprints = Dict{String,Any}()
    for rank in 0:1
        bytime = Dict{Float64,NamedTuple}()
        for path in parts[rank]
            before = fingerprint(path)
            jldopen(path, "r"; iotype=IOStream) do file
                haskey(file, "timeseries/t") || error("No timeseries/t in $path")
                selected_keys = String.(collect(keys(file["timeseries/t"])))
                for key in selected_keys
                    isnothing(tryparse(Int, key)) && continue
                    time = Float64(file["timeseries/t/$key"])
                    isfinite(time) && time >= 0 || error("Invalid CHL timestamp in $path")
                    time <= cutoff_time || continue
                    if !haskey(file, "timeseries/chl/$key")
                        push!(issues, (; run=run.name, rank, path, model_day=time/day_seconds,
                            key, status="missing chl", detail="Saved chl is required; P is not substituted"))
                        continue
                    end
                    if haskey(bytime, time)
                        previous = bytime[time]
                        previous.key == key || error("Conflicting CHL iterations at day $(time/day_seconds) in $(run.name), rank $rank")
                        push!(issues, (; run=run.name, rank, path, model_day=time/day_seconds,
                            key, status="duplicate part boundary", detail=previous.path))
                    end
                    bytime[time] = (; time, key, path)
                end
            end
            input_mode == :strict && assert_unchanged(path, before)
            fingerprints[path] = before
        end
        tables[rank] = sort(collect(values(bytime)); by=r -> r.time)
        all(diff([r.time for r in tables[rank]]) .> 1.0) ||
            error("Ambiguous CHL timestamps within $(run.name), rank $rank")
    end
    records, used = NamedTuple[], Set{Int}()
    j = 1
    for a in tables[0]
        while j <= length(tables[1]) && tables[1][j].time < a.time-1.0
            j += 1
        end
        if j <= length(tables[1]) && abs(tables[1][j].time-a.time) <= 1.0
            b = tables[1][j]
            a.key == b.key || error("CHL rank iterations differ at day $(a.time/day_seconds) in $(run.name)")
            push!(records, (; time=a.time, keys=(a.key,b.key), paths=(a.path,b.path)))
            push!(used, j)
            j += 1
        else
            push!(issues, (; run=run.name, rank=0, path=a.path, model_day=a.time/day_seconds,
                key=a.key, status="missing paired rank", detail="rank 1"))
        end
    end
    for (i,b) in enumerate(tables[1])
        i in used && continue
        push!(issues, (; run=run.name, rank=1, path=b.path, model_day=b.time/day_seconds,
            key=b.key, status="missing paired rank", detail="rank 0"))
    end
    return (; records, fingerprints)
end


end


Main.ReferenceChlComparison

In [13]:
const parameter_names = ["maximum_growth_rate", "chlorophyll_ratio", "iron_half_saturation",
    "nitrate_half_saturation", "light_half_saturation", "maximum_grazing_rate"]
const parameter_units = ["day^-1", "mg Chl (mmol N)^-1", "nM", "uM", "W m^-2", "day^-1"]
const native_parameter_names = ["phytoplankton_maximum_growth_rate", "chlorophyll_ratio",
    "iron", "nitrate", "light_half_saturation", "maximum_grazing_rate"]
const parameter_factors = [86400.0, 1.0, 1000.0, 1.0, 1.0, 86400.0]

# Read only known numeric assignments. Never evaluate text from experiment.txt.
function recorded_parameters(chain)
    values = fill(NaN, 6)
    number = raw"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eEfF][+-]?\d+)?)"
    for run in chain, key in ("resolved_test_parameters", "plankton_parameters")
        text = get(run.metadata, key, "")
        for j in eachindex(values)
            pattern = Regex(raw"(?:^|[\s,(])" * native_parameter_names[j] * raw"\s*=\s*" * number * raw"(?=\s*[,)]|\s*$)")
            found = match(pattern, text)
            isnothing(found) && continue
            value = parse(Float64, replace(lowercase(found.captures[1]), "f" => "e")) * parameter_factors[j]
            if isfinite(values[j]) && !isapprox(values[j], value; rtol=1e-6, atol=1e-10)
                error("Conflicting recorded $(parameter_names[j]) in $(run.name)")
            end
            values[j] = value
        end
    end
    return values
end

function parameter_table(path, ids, chains; reference_defaults=nothing)
    recorded = [recorded_parameters(chains[id]) for id in ids]
    if !isfile(path)
        open(path, "w") do io
            println(io, join(vcat(["sample_id"], parameter_names), ","))
            for (id, row) in zip(ids, recorded)
                println(io, join(vcat([id], [isfinite(v) ? string(v) : "" for v in row]), ","))
            end
        end
        println("Created parameter table: ", path)
    end
    rows = split.(filter(x -> !isempty(strip(x)), readlines(path)), ','; keepempty=true)
    strip.(first(rows)) == vcat(["sample_id"], parameter_names) || error("Unexpected pilot parameter CSV header")
    table = Dict{String,Vector{Float64}}()
    changed = false
    existing_ids = Set(strip(row[1]) for row in rows[2:end])
    for (id, values) in zip(ids, recorded)
        if !(id in existing_ids)
            push!(rows, vcat([id], [isfinite(value) ? string(value) : "" for value in values]))
            changed = true
        end
    end
    for row in rows[2:end]
        length(row) == 7 || error("Pilot parameter CSV needs seven columns")
        id = strip(row[1])
        haskey(table, id) && error("Duplicate sample_id: $id")
        values = [something(tryparse(Float64, strip(v)), NaN) for v in row[2:end]]
        if id == "reference" && id in ids && !isnothing(reference_defaults)
            position = findfirst(==("reference"), ids)
            isnothing(position) && error("CSV includes an unselected reference case")
            for (j, name) in ((2, :chlorophyll_ratio), (5, :light_half_saturation))
                # Fill empty Reference entries only when the metadata omits the parameter.
                if !isfinite(recorded[position][j]) && isempty(strip(row[j+1]))
                    values[j] = Float64(getproperty(reference_defaults, name))
                    row[j+1] = string(values[j])
                    changed = true
                end
            end
        end
        table[id] = values
    end
    missing_ids = setdiff(ids, collect(keys(table)))
    isempty(missing_ids) || error("Selected cases missing from pilot CSV: $(join(missing_ids, ", "))")
    for (id, known) in zip(ids, recorded), j in eachindex(known)
        value = table[id][j]
        isfinite(value) && value > 0 || error("Fill $(parameter_names[j]) for $id in $path from that run's actual configuration. Other missing parameters require values from the simulation configuration.")
        !isfinite(known[j]) || isapprox(value, known[j]; rtol=1e-6, atol=1e-10) ||
            error("CSV $(parameter_names[j]) for $id conflicts with experiment.txt")
    end
    if changed
        open(path, "w") do io
            for row in rows
                println(io, join(row, ","))
            end
        end
        println("Updated pilot parameter table with selected cases and reference defaults.")
    end
    return reduce(vcat, [permutedims(table[id]) for id in ids])
end

function selected_chain(output_root, id, leaf)
    found = BiologyCatalog.discover_biology_runs(output_root, [id])
    haskey(found.runs, leaf) || error("Selected run missing/invalid: $leaf. Metadata errors: $(found.invalid)")
    return BiologyCatalog.resolve_biology_lineage(found.runs, leaf)
end

function native_geometry(files, records, variable)
    geometries = Dict{Tuple{String,String},Any}()
    reference = [BiologyAnalysis.file_geometry(files, first(records).paths[k], variable, geometries) for k in 1:2]
    reference[1].nx == reference[2].nx || error("Expected two ranks partitioned in latitude")
    !(reference[1].lon == reference[2].lon && reference[1].lat == reference[2].lat) || error("Duplicate rank grids")
    for record in records, rank in 1:2
        BiologyAnalysis.verify_geometry(reference[rank], BiologyAnalysis.file_geometry(files, record.paths[rank], variable, geometries))
    end
    areas = [vec([Float64(Oceananigans.Operators.Azᶜᶜᶠ(i,j,g.nz+1,g.grid))
        for i in 1:g.nx, j in 1:g.ny]) for g in reference]
    geometry = (; lon=vcat([g.lon for g in reference]...), lat=vcat([g.lat for g in reference]...),
        wet=vcat([g.wet for g in reference]...), area=vcat(areas...))
    all(isfinite.(geometry.area[geometry.wet]) .& (geometry.area[geometry.wet] .> 0)) || error("Invalid wet-cell area")
    return geometry, geometries
end

function pilot_fields(chain, start_day, end_day; chemistry=OceanBioME.CarbonChemistry(Float64))
    pp_by_run = Dict(run.name => DailyPPReader.index_daily_run(run; input_mode=:fixed_records,
        cutoff_day=end_day) for run in chain)
    pp_catalog = (; records=BiologyCatalog.stitch_biology_records(chain, pp_by_run))
    pp_records = DailyPPReader.window_records(pp_catalog, start_day, end_day)
    issues = NamedTuple[]
    chl_by_run = Dict(run.name => ReferenceChlComparison.index_chl_run(run, issues;
        input_mode=:fixed_records, cutoff_day=end_day).records for run in chain)
    chl_records = BiologyCatalog.stitch_biology_records(chain, chl_by_run)
    chl_records = BiologyChlorophyll.window_records(chl_records, start_day*86400.0, end_day*86400.0)
    pp_geometry, pp = BiologyAnalysis.with_input_files(pp_records) do files
        geometry, cache = native_geometry(files, pp_records, "PP_C_full_column")
        for (path, file) in files
            DailyPPReader.verify_metadata(file, path; variable="PP_C_full_column")
        end
        function read_plane(record)
            parts = Vector{Float64}[]
            for rank in 1:2
                path, key = record.paths[rank], record.keys[rank]
                file = files[path]
                abs(Float64(file["timeseries/t/$key"])-record.time) <= 1.0 || error("Selected PP timestamp changed")
                g = BiologyAnalysis.file_geometry(files, path, "PP_C_full_column", cache)
                qg = BiologyAnalysis.file_geometry(files, path, "PP_invalid_fraction", cache)
                BiologyAnalysis.verify_geometry(g, qg)
                values = DailyPPReader.plane_values(file, "PP_C_full_column", key, g)
                quality = DailyPPReader.plane_values(file, "PP_invalid_fraction", key, qg)
                values[.!g.wet .| (quality .!= 0) .| .!isfinite.(values) .| (values .< 0)] .= NaN
                push!(parts, values)
            end
            return vcat(parts...)
        end
        return geometry, DailyPPReader.average_planes(read_plane, pp_records)
    end
    chl_geometry, combined = BiologyAnalysis.with_input_files(chl_records) do files
        geometry, cache = native_geometry(files, chl_records, "chl")
        function read_snapshot(record)
            chl_parts, fco2_parts = Vector{Float64}[], Vector{Float64}[]
            for rank in 1:2
                path, key = record.paths[rank], record.keys[rank]
                file = files[path]
                abs(Float64(file["timeseries/t/$key"])-record.time) <= 1.0 || error("Selected surface timestamp changed")
                required = ("chl", "DIC1", "Alk1", "T", "S")
                for variable in required
                    haskey(file, "timeseries/$variable/$key") ||
                        error("Missing $variable required for control fCO2 at $path, iteration $key")
                end
                g = BiologyAnalysis.file_geometry(files, path, "chl", cache)
                values = Dict{String,Vector{Float64}}()
                for variable in required
                    vg = BiologyAnalysis.file_geometry(files, path, variable, cache)
                    BiologyAnalysis.verify_geometry(g, vg)
                    values[variable] = BiologyAnalysis.surface_values(file, variable, key, vg)
                end
                chl = values["chl"]
                chl[.!g.wet .| .!isfinite.(chl) .| (chl .< 0)] .= NaN
                fco2 = control_fco2(values["DIC1"], values["Alk1"], values["T"], values["S"], g.wet, chemistry)
                push!(chl_parts, chl)
                push!(fco2_parts, fco2)
            end
            # One streaming pass reads each saved snapshot for both diagnostics.
            # Diagnose fCO2 cell by cell before averaging; chemistry is nonlinear.
            return vcat(vcat(chl_parts...), vcat(fco2_parts...))
        end
        return geometry, BiologyChlorophyll.stream_month_mean(read_snapshot, chl_records,
            start_day*86400.0, end_day*86400.0)
    end
    n = length(chl_geometry.wet)
    chl, fco2 = combined[1:n], combined[n+1:end]
    same_geometry(pp_geometry, chl_geometry)
    for run in chain
        BiologyCatalog.read_biology_metadata(joinpath(run.directory,"experiment.txt")) == run.metadata || error("Metadata changed during export")
    end
    return (; geometry=pp_geometry, CHL=chl, PP=pp, fCO2=fco2)
end

function same_geometry(a, b)
    a.wet == b.wet && isapprox(a.lon,b.lon;atol=1e-6,rtol=0) &&
        isapprox(a.lat,b.lat;atol=1e-6,rtol=0) && isapprox(a.area,b.area;rtol=1e-8) ||
        error("Selected model cases or variables have different native grids")
end

function check_reconstruction_source(text)
    nodes = Expr[]
    function visit(node)
        node isa Expr || return
        node.head in (:quote,:inert) && return
        push!(nodes,node)
        foreach(visit,node.args)
    end
    visit(Meta.parseall(String(text)))
    assignments = filter(nodes) do node
        node.head == :(=) || return false
        lhs=node.args[1]
        lhs==:carbon_chemistry || (lhs isa Expr && lhs.head==:(::) && lhs.args[1]==:carbon_chemistry)
    end
    length(assignments)==1 || error("Reconstruction requires one assignment to carbon_chemistry")
    rhs=only(assignments).args[2]
    rhs isa Expr && rhs.head==:call && rhs.args[1]==:CarbonChemistry &&
        (length(rhs.args)==1 || (length(rhs.args)==2 && rhs.args[2]==:Float64)) ||
        error("fCO2 reconstruction requires the default CarbonChemistry constructor")
    any(nodes) do node
        node.head==:(=) && node.args[1] isa Expr && node.args[1].head==:. &&
            node.args[1].args[1]==:carbon_chemistry
    end && error("Carbon chemistry fields are modified after construction; reconstruction defaults cannot be assumed")
    constructors=filter(node -> node.head==:call && node.args[1]==:CarbonDioxideConcentration,nodes)
    mappings=Tuple{Symbol,Symbol}[]
    for constructor in constructors
        parameters=filter(x -> x isa Expr && x.head==:parameters,constructor.args)
        length(parameters)==1 || continue
        kwargs=Dict{Symbol,Any}()
        for argument in only(parameters).args
            argument isa Symbol && (kwargs[argument]=argument)
            argument isa Expr && argument.head==:kw && (kwargs[argument.args[1]]=argument.args[2])
        end
        haskey(kwargs,:silicate_and_phosphate_names) &&
            error("Model specifies carbonate nutrient terms; the zero-phosphate/silicate reconstruction does not apply")
        get(kwargs,:carbon_chemistry,nothing)==:carbon_chemistry ||
            error("CarbonDioxideConcentration does not use the carbon_chemistry binding")
        dic,alk=get(kwargs,:DIC,nothing),get(kwargs,:Alk,nothing)
        dic isa QuoteNode && alk isa QuoteNode && push!(mappings,(dic.value,alk.value))
    end
    for pair in ((:DIC1,:Alk1),(:DIC2,:Alk2))
        count(==(pair),mappings)==1 || error("Cannot establish exactly one CarbonDioxideConcentration mapping for $pair")
    end
    return nothing
end


function pilot_chemistry_provenance(path, chains)
    isfile(path) || error("Set chemistry_source to the Biogeochemistry source used by these runs: $path")
    check_reconstruction_source(read(path, String))
    version = Base.pkgversion(OceanBioME)
    version == v"0.18.0" || error("Expected OceanBioME v0.18.0; active version: $version")
    for chain in values(chains), run in chain
        recorded = get(run.metadata, "OceanBioME_version", string(version))
        recorded == string(version) || error("OceanBioME version differs from recorded run $(run.name): $recorded")
    end
    package_root = Base.pkgdir(OceanBioME)
    gas_path = joinpath(package_root, "src", "Models", "GasExchange", "carbon_dioxide_concentration.jl")
    chemistry_path = joinpath(package_root, "src", "Models", "CarbonChemistry", "carbon_chemistry.jl")
    compact(text) = replace(replace(text, r"#[^\n]*" => ""), r"\s+" => "")
    gas = compact(read(gas_path, String))
    occursin("output=Val(:pCO₂)", gas) && occursin("silicate_and_phosphate(::Nothing", gas) &&
        occursin("=(0,0)", gas) || error("Installed gas-exchange nutrient/chemistry settings differ from the supported defaults")
    occursin("selected_output(::Val{:fCO₂}", compact(read(chemistry_path, String))) ||
        error("Installed CarbonChemistry does not expose the direct fCO2 output")
    return (; source_file=abspath(path), oceanbiome_version=string(version),
        description="Control DIC1/Alk1/T/S, CarbonChemistry(Float64), output=Val(:fCO₂), phosphate=0, silicate=0, default surface pressure; per-snapshot before time averaging")
end

function control_fco2(dic, alk, temperature, salinity, wet, chemistry)
    output = fill(NaN, length(wet))
    Threads.@threads for i in eachindex(wet)
        wet[i] || continue
        if isfinite(dic[i]) && dic[i] > 0 && isfinite(alk[i]) && alk[i] > 0 &&
                isfinite(temperature[i]) && temperature[i] > -273.15 &&
                isfinite(salinity[i]) && salinity[i] >= 0
            value = chemistry(; DIC=Float64(dic[i]), Alk=Float64(alk[i]),
                T=Float64(temperature[i]), S=Float64(salinity[i]),
                phosphate=0.0, silicate=0.0, output=Val(:fCO₂))
            isfinite(value) && value > 0 || error("Nonfinite/nonpositive fCO2 at wet cell $i")
            output[i] = value
        end
    end
    return output
end


control_fco2 (generic function with 1 method)

## 3. Case parameters

Create `data/pilot_parameters.csv` from parameters recorded in each run's `experiment.txt`.
Growth and grazing rates are converted from s⁻¹ to day⁻¹; iron half-saturation is converted from mmol m⁻³ to nM.

Missing Reference chlorophyll and light parameters use `reference_defaults`: 1.31 and 33.0.
Recorded values take precedence, and populated CSV entries are preserved. Other missing entries require
values from the corresponding simulation configuration.

The carbonate calculation uses the model's default `CarbonChemistry(Float64)` configuration with zero
phosphate and silicate contributions. The source and package checks below enforce this configuration.
Only the cases selected in Section 1 are exported. Newly selected cases are added from metadata; other CSV rows are retained for later use.


In [14]:
period_start >= day_zero && period_end >= period_start || error("Invalid requested period")
start_day = Dates.value(period_start-day_zero)
end_day = Dates.value(period_end+Day(1)-day_zero)
ids = [case.id for case in cases]
length(unique(ids)) == length(ids) || error("Duplicate selected case IDs")
chains = Dict(case.id => selected_chain(joinpath(model_root,case.folder),case.id,case.leaf) for case in cases)
parameter_path = joinpath(data_dir,"pilot_parameters.csv")
parameters = parameter_table(parameter_path, ids, chains; reference_defaults=reference_defaults)
chemistry_provenance = pilot_chemistry_provenance(chemistry_source, chains)
chemistry = OceanBioME.CarbonChemistry(Float64)
println("fCO2: ", chemistry_provenance.description)
println("Selected ", length(ids), " independent cases; continuations are merged into their original cases.")
println("Period: ", period_start, " through ", period_end, " (", end_day-start_day, " days)")


fCO2: Control DIC1/Alk1/T/S, CarbonChemistry(Float64), output=Val(:fCO₂), phosphate=0, silicate=0, default surface pressure; per-snapshot before time averaging
Selected 4 independent cases; continuations are merged into their original cases.
Period: 2000-01-01 through 2001-12-31 (731 days)


## 4. Export mean fields

Compute CHL, PP and control fCO2 over the configured period on a common finite ocean mask.
The NetCDF file stores the mean fields, cell areas, coordinates, case parameters and source information.
PP is averaged from daily interval means; CHL and fCO2 use trapezoidal time integration.

The record checks require complete period coverage. For a shorter test, change the dates in Section 1
and use the same `pilot_period` in the NN notebook.

In [15]:
# Average PP daily means and integrate CHL/fCO2 snapshots over time.
# Require complete records and matching grids.
fields = map(ids) do id
    println("Preparing case: ", id)
    try
        pilot_fields(chains[id],start_day,end_day; chemistry)
    catch err
        err isa InterruptException && rethrow()
        error("Case $id: $(sprint(showerror, err))")
    end
end
geometry = first(fields).geometry
foreach(field -> same_geometry(geometry,field.geometry),fields)
common = copy(geometry.wet)
for field in fields
    common .&= isfinite.(field.CHL) .& isfinite.(field.PP) .& isfinite.(field.fCO2)
end
any(common) || error("No common valid native cells in the selected cases and period")
println("Common valid native cells: ", count(common))

output_path = joinpath(data_dir,"pilot_2000-2001.nc")
temporary_path = output_path * ".part"
NCDataset(temporary_path,"c") do ds
    defDim(ds,"sample",length(ids))
    defDim(ds,"feature",count(common))
    defVar(ds,"sample_id",String,("sample",))[:] = ids
    for j in eachindex(parameter_names)
        variable = defVar(ds,parameter_names[j],Float64,("sample",),attrib=Dict("units"=>parameter_units[j]))
        variable[:] = parameters[:,j]
    end
    for (target,unit) in (("CHL","mg Chl m^-3"),("PP","mg C m^-2 day^-1"),("fCO2","uatm"))
        # Julia writes dimensions in reverse storage order. Defining feature,sample
        # produces the required Python/NetCDF order sample,feature.
        variable = defVar(ds,target,Float32,("feature","sample"),attrib=Dict("units"=>unit),deflatelevel=2)
        variable[:,:] = hcat([Float32.(getproperty(field,Symbol(target))[common]) for field in fields]...)
    end
    defVar(ds,"cell_area",Float64,("feature",),attrib=Dict("units"=>"m^2"))[:] = geometry.area[common]
    defVar(ds,"longitude",Float64,("feature",),attrib=Dict("units"=>"degrees_east"))[:] = geometry.lon[common]
    defVar(ds,"latitude",Float64,("feature",),attrib=Dict("units"=>"degrees_north"))[:] = geometry.lat[common]
    defVar(ds,"source_runs",String,("sample",))[:] = [join([run.name for run in chains[id]]," -> ") for id in ids]
    ds.attrib["period_start"] = string(period_start)
    ds.attrib["period_end"] = string(period_end)
    ds.attrib["time_origin"] = string(day_zero)
    ds.attrib["purpose"] = "Pilot ensemble for emulator development"
    ds.attrib["CHL_definition"] = "Saved surface OceanBioME chlorophyll; actual-time trapezoidal mean"
    ds.attrib["PP_definition"] = "PP_C_full_column; complete daily mean integrated over native wet water column"
    ds.attrib["fCO2_definition"] = chemistry_provenance.description
    ds.attrib["carbon_chemistry_source"] = chemistry_provenance.source_file
    ds.attrib["OceanBioME_version"] = chemistry_provenance.oceanbiome_version
    ds.attrib["fCO2_units"] = "uatm"
    ds.attrib["carbon_nutrients"] = "phosphate=0 and silicate=0, matching the model gas-exchange configuration"
    ds.attrib["carbon_pressure"] = "CarbonChemistry default surface-pressure convention (P=nothing)"
    ds.attrib["parameter_source"] = "experiment.txt and configured Reference defaults; case values stored in pilot_parameters.csv"
end
mv(temporary_path,output_path;force=true)
println("Saved ", output_path)
println("Set pilot_period = ", repr((string(period_start),string(period_end))), " in the Python notebook.")
println("Next: use run_mode = \"pilot\" in oae_nn_emulator.ipynb.")


Preparing case: reference
Preparing case: chl30_kfe50
Preparing case: chl30_growth_plus25
Preparing case: chl30_kfe50_growth_plus25
Common valid native cells: 40569
Saved /nfs/roberts/pi/pi_ey239/qz334/ML4OAE/data/pilot_2000-2001.nc
Set pilot_period = ("2000-01-01", "2001-12-31") in the Python notebook.
Next: use run_mode = "pilot" in oae_nn_emulator.ipynb.
